# Fase 3a — Estrategia 0: macro-contextos y partición Train/Test

Punto de partida de la construcción de datasets de train y test: toma el corpus limpio en Markdown de la Fase 2 (`corpus_rag_final_gemini.json`) y genera, por documento, **macro-secciones** de gran tamaño (~4000 caracteres, sin solapamiento) usando `RecursiveCharacterTextSplitter` de LangChain. Estos macro-contextos son la unidad sobre la que después se generarán las preguntas (Fase 4).

Pasos del notebook:

1. **Generación de macro-contextos** (`generar_macro_contextos_langchain`): trocea el texto de cada documento respetando preferentemente los párrafos, descartando fragmentos demasiado cortos (<100 caracteres).
2. **Partición Train/Test estratificada** (`preparar_y_guardar_datasets`): el split 80/20 se aplica documento a documento (no sobre el conjunto global), de forma que cada PDF aporta proporcionalmente a ambos conjuntos y ningún documento queda fuera de alguno de los dos.
3. **Auditoría de granularidad** (`auditar_macro_contextos`): calcula longitud media/mediana/máxima/mínima de los macro-contextos generados y valida que conservan los campos esperados.
4. **Análisis de distribución por documento** (`analizar_distribucion_por_documento`): muestra cuántos macro-contextos de cada documento han caído en train y en test, informando de repartos problemáticos (documento ausente en alguno de los dos conjuntos, o split muy desequilibrado).

In [4]:
!pip install langchain langchain-text-splitters

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid

def generar_macro_contextos_langchain(doc_json, chunk_size=4000):
    """
    Genera los Macro-Contextos base para Train/Test usando LangChain.
    """
    texto = doc_json["page_content"]

    # 1. Configuramos el splitter SIN SOLAPAMIENTO y TAMAÑO GIGANTE
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=0,          # Cero fuga de datos.
        separators=["\n\n", "\n", ". ", " ", ""] # Solo cortamos por párrafos, no queremos cortar a mitad de frase.
    )

    textos_troceados = splitter.split_text(texto)
    macro_contextos = []

    for texto_chunk in textos_troceados:
        if len(texto_chunk.strip()) < 100: # Filtramos chunks muy cortos
            continue

        macro_contextos.append({
            "macro_id": str(uuid.uuid4()), # ID único para separar luego en Train/Test
            "source": doc_json["metadata"].get("source", "desconocido"),
            "text": texto_chunk
        })

    return macro_contextos

In [6]:
import random
import json
import os

def preparar_y_guardar_datasets(CORPUS_FILE, ruta_salida_train="dataset_train.json", ruta_salida_test="dataset_test.json", porcentaje_train=0.8):
    """
    Coge la lista completa de tus PDFs, genera los Macro-Contextos
    y hace un split ESTRATIFICADO (80/20 por cada documento individual).
    """

    with open(CORPUS_FILE, 'r', encoding='utf-8') as f:
        corpus = json.load(f)

    dataset_train = []
    dataset_test = []

    random.seed(42) # Semilla global para reproducibilidad

    # Procesamos y dividimos PDF por PDF
    for doc_json in corpus:
        # 1. Generamos los macro-contextos solo de este PDF
        macros_de_este_doc = generar_macro_contextos_langchain(doc_json, chunk_size=4000)

        # 2. Mezclamos los fragmentos de este PDF en concreto
        random.shuffle(macros_de_este_doc)

        # 3. Calculamos el corte para ESTE PDF
        corte = int(len(macros_de_este_doc) * porcentaje_train)

        # 4. Separamos en Train y Test
        train_doc = macros_de_este_doc[:corte]
        test_doc = macros_de_este_doc[corte:]

        # 5. Los añadimos a las listas globales
        dataset_train.extend(train_doc)
        dataset_test.extend(test_doc)

        # Opcional: imprimir cómo se ha dividido cada documento para estar seguros
        source_name = doc_json["metadata"].get("source", "desconocido")
        # print(f"Doc '{source_name}': {len(train_doc)} a Train, {len(test_doc)} a Test")

    with open(ruta_salida_train, 'w', encoding='utf-8') as f:
        json.dump(dataset_train, f, ensure_ascii=False, indent=4)

    with open(ruta_salida_test, 'w', encoding='utf-8') as f:
        json.dump(dataset_test, f, ensure_ascii=False, indent=4)

    print(f"¡Éxito! Archivos guardados en:\n- {ruta_salida_train}\n- {ruta_salida_test}")

    print(f"Total de Macro-Contextos para Entrenamiento (Train): {len(dataset_train)}")
    print(f"Total de Macro-Contextos para Evaluación (Test): {len(dataset_test)}")

    return dataset_train, dataset_test

In [8]:
# ==========================================
# EJECUCIÓN
# ==========================================

dataset_train, dataset_test = preparar_y_guardar_datasets('corpus_rag_final_gemini.json')

¡Éxito! Archivos guardados en:
- dataset_train.json
- dataset_test.json
Total de Macro-Contextos para Entrenamiento (Train): 275
Total de Macro-Contextos para Evaluación (Test): 78


In [9]:
import json
import statistics
import random

ARCHIVOS_MACRO = {
    "Macro-Contextos (Train)": 'dataset_train.json',
    "Macro-Contextos (Test)": 'dataset_test.json'
}

def calcular_metricas(lista_macros):
    """Función auxiliar para calcular e imprimir métricas de los bloques gigantes"""
    longitudes = [len(c["text"]) for c in lista_macros]

    if not longitudes:
        print(" -> El dataset está vacío.")
        return

    print(f" -> Total de bloques:         {len(lista_macros)}")
    print(f" -> Longitud Media:           {statistics.mean(longitudes):.0f} caracteres")
    print(f" -> Longitud Mediana:         {statistics.median(longitudes):.0f} caracteres")
    print(f" -> Longitud Máxima:          {max(longitudes)} caracteres")
    print(f" -> Longitud Mínima:          {min(longitudes)} caracteres")

def auditar_macro_contextos():
    """
    Carga los datasets de macro-contextos de train y test, calcula sus
    métricas de granularidad (calcular_metricas) y realiza un control de
    calidad básico: comprueba que cada bloque tiene las claves esperadas
    (macro_id, source, text) y muestra una muestra aleatoria de texto para
    inspección visual.
    """
    print("INICIANDO AUDITORÍA DE GRANULARIDAD DE LOS DATASETS BASE")
    print("="*75)

    for nombre, archivo in ARCHIVOS_MACRO.items():
        try:
            with open(archivo, 'r', encoding='utf-8') as f:
                macros = json.load(f)
        except FileNotFoundError:
            print(f"\n[X] Archivo no encontrado: {archivo}")
            continue

        if not macros:
            print(f"\n{nombre}: El archivo está vacío.")
            continue

        print(f"\n{nombre.upper()}")
        print("-" * 50)
        print("   [ MÉTRICAS DE GRANULARIDAD ]")

        # Calculamos la longitud de nuestros bloques gigantes
        calcular_metricas(macros)

        # --- VALIDACIONES DE CALIDAD ---
        print("\n   [ CONTROL DE CALIDAD Y ESTRUCTURA ]")

        # Comprobamos que tienen las claves que necesitamos para la Fase 2
        macro_prueba = macros[0] if macros else {}
        claves_base = ["macro_id", "source", "text"]
        claves_ok = all(k in macro_prueba for k in claves_base)
        print(f"   -> Integridad de claves base: {'OK' if claves_ok else 'ERROR: Faltan campos clave'}")

        # Muestra visual para ver de qué documento viene
        muestra = random.choice(macros)
        texto_muestra = muestra['text'].replace('\n', ' ')[:100]
        origen = muestra.get("source", "Desconocido")

        print(f"   -> Muestra al azar (Origen: {origen}):")
        print(f"      \"{texto_muestra}...\"")
        print("="*75)

if __name__ == "__main__":
    auditar_macro_contextos()

INICIANDO AUDITORÍA DE GRANULARIDAD DE LOS DATASETS BASE

MACRO-CONTEXTOS (TRAIN)
--------------------------------------------------
   [ MÉTRICAS DE GRANULARIDAD ]
 -> Total de bloques:         275
 -> Longitud Media:           3212 caracteres
 -> Longitud Mediana:         3563 caracteres
 -> Longitud Máxima:          3992 caracteres
 -> Longitud Mínima:          284 caracteres

   [ CONTROL DE CALIDAD Y ESTRUCTURA ]
   -> Integridad de claves base: OK
   -> Muestra al azar (Origen: 16264_Vacunación estacional frente a infecciones respiratoria (gripe y COVID-19). Temporada 2024-2025.pdf):
      "La población de referencia por grupos de edad, tanto en el grupo de 60 o más años como en la poblaci..."

MACRO-CONTEXTOS (TEST)
--------------------------------------------------
   [ MÉTRICAS DE GRANULARIDAD ]
 -> Total de bloques:         78
 -> Longitud Media:           3086 caracteres
 -> Longitud Mediana:         3444 caracteres
 -> Longitud Máxima:          3990 caracteres
 -> Longitud 

In [10]:
import json
from collections import defaultdict

def analizar_distribucion_por_documento(ruta_train='dataset_train.json', ruta_test='dataset_test.json'):
    """
    Agrupa los macro-contextos de train y test por documento de origen y
    muestra una tabla con el número de bloques de cada conjunto por
    documento, señalando avisos cuando un documento queda sin representación
    en train o en test, o cuando el reparto resulta muy desigual (p. ej.
    un split 50/50 por tener muy pocos bloques).
    """
    print("INICIANDO ANÁLISIS DE DISTRIBUCIÓN POR DOCUMENTO")
    print("="*80)

    try:
        with open(ruta_train, 'r', encoding='utf-8') as f:
            train_data = json.load(f)
        with open(ruta_test, 'r', encoding='utf-8') as f:
            test_data = json.load(f)
    except Exception as e:
        print(f"Error al cargar los archivos: {e}")
        return

    # Diccionarios para guardar el recuento y longitudes por documento
    # Estructura: {"nombre_doc": {"train": [len1, len2...], "test": [len1...]}}
    distribucion = defaultdict(lambda: {"train": [], "test": []})

    for bloque in train_data:
        origen = bloque.get("source", "Desconocido")
        distribucion[origen]["train"].append(len(bloque["text"]))

    for bloque in test_data:
        origen = bloque.get("source", "Desconocido")
        distribucion[origen]["test"].append(len(bloque["text"]))

    # Imprimir la tabla de resultados
    print(f"{'DOCUMENTO (SOURCE)':<50} | {'TRAIN':<10} | {'TEST':<10} | {'AVISOS'}")
    print("-" * 80)

    for doc, datos in sorted(distribucion.items()):
        num_train = len(datos["train"])
        num_test = len(datos["test"])

        # Generar alertas si hay desbalanceos graves
        alerta = ""
        if num_train == 0:
            alerta = "CERO en Train (Peligro: OOD)"
        elif num_test == 0:
            alerta = "CERO en Test (No se evaluará)"
        elif num_train == 1 and num_test == 1:
            alerta = "Split 50/50"

        # Acortamos el nombre del documento si es muy largo para que la tabla no se rompa
        nombre_corto = doc[:47] + "..." if len(doc) > 50 else doc

        print(f"{nombre_corto:<50} | {num_train:<10} | {num_test:<10} | {alerta}")

    print("="*80)
    print(f"Total de documentos únicos detectados: {len(distribucion)}")

if __name__ == "__main__":
    analizar_distribucion_por_documento()

INICIANDO ANÁLISIS DE DISTRIBUCIÓN POR DOCUMENTO
DOCUMENTO (SOURCE)                                 | TRAIN      | TEST       | AVISOS
--------------------------------------------------------------------------------
10584_Vacunación estacional frente a infeccione... | 19         | 5          | 
10604_Protocolo de uso de Nivolumab en condicio... | 2          | 1          | 
10605_Protocolo de uso Fuera de Ficha Técnica d... | 3          | 1          | 
10624_Protocolo de uso de Pembrolizumab en cond... | 3          | 1          | 
11064_Protocolo de vigilancia, prevención y con... | 20         | 6          | 
1181_Guía hospitalaria de terapéutica antibióti... | 50         | 13         | 
11864_Procedimiento para el cumplimiento de la ... | 2          | 1          | 
13564_Logística vacunal. Cadena de frío.pdf        | 4          | 2          | 
13584_Protocolo de vigilancia de la enfermedad ... | 8          | 2          | 
15264_Protocolo Regional para la indicación, us... | 24         

In [11]:
from google.colab import files
import os
import zipfile

# Lista de tus archivos de salida
filenames = [
    'dataset_train.json',
    'dataset_test.json'
]

zip_name = 'dataset_evaluacion_train_test.zip'

print(f"Comprimiendo archivos en {zip_name}...")

with zipfile.ZipFile(zip_name, 'w') as zipf:
    found_any = False
    for file in filenames:
        if os.path.exists(file):
            print(f"  -> Añadiendo: {file}")
            zipf.write(file)
            found_any = True
        else:
            print(f"  Warning: No se encontró {file} (quizás no se generó)")

if found_any:
    print("¡Descargando ZIP!")
    files.download(zip_name)
else:
    print("No se encontraron archivos para descargar.")

Comprimiendo archivos en dataset_evaluacion_train_test.zip...
  -> Añadiendo: dataset_train.json
  -> Añadiendo: dataset_test.json
¡Descargando ZIP!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>